## Some dependencies might require installationm specifically zellkonverter and CARD

```
devtools::install_github('YingMa0107/CARD')
BiocManager::install("zellkonverter")
```

In [ ]:
suppressPackageStartupMessages({
    library(Seurat)
    library(SeuratData)
    library(SeuratDisk)
    library(SCP)
    library(ggplot2)
    library(patchwork)
    library(dplyr)
    library(Matrix)
    library(rhdf5)
    library(CARD)
    library(zellkonverter)
    library(SingleCellExperiment)
})

In [ ]:
## Helper function

convert_CARD_to_AnnData <- function(obj, out_file) {
  if (inherits(obj, "CARD")) {
    message("Converting a CARD object to SingleCellExperiment...")
    
    counts_mat <- obj@spatial_countMat
    coords_df  <- obj@spatial_location
    prop_mat   <- obj@Proportion_CARD
    
    if (!identical(colnames(counts_mat), rownames(prop_mat))) {
      message("Transposing cell type proportions to match counts matrix columns...")
      prop_mat <- t(prop_mat)
    }
    
    if (!is.null(rownames(coords_df)) && !identical(rownames(coords_df), colnames(counts_mat))) {
      message("Reordering coordinates to match the spot order in counts_mat...")
      coords_df <- coords_df[colnames(counts_mat), , drop = FALSE]
    }
    
    if (!inherits(counts_mat, "sparseMatrix")) {
      counts_mat <- as(counts_mat, "dgCMatrix")
    }
    
    sce <- SingleCellExperiment(
      assays = list(
        counts = counts_mat
      )
    )
    
    prop_df <- as.data.frame(prop_mat)
    colnames(prop_df) <- make.names(colnames(prop_df))
    rownames(prop_df) <- colnames(counts_mat)
    colData(sce) <- DataFrame(prop_df)
    
    colData(sce)$pos_x <- coords_df[, 1]
    colData(sce)$pos_y <- coords_df[, 2]
    reducedDims(sce)$spatial <- as.matrix(coords_df[, 1:2])
    
    writeH5AD(sce, out_file)
    
  } else if (inherits(obj, "SingleCellExperiment")) {
    message("Object is already a SingleCellExperiment. Writing AnnData directly...")
    writeH5AD(obj, out_file)
    
  } else {
    warning("The object is neither a CARD object nor a SingleCellExperiment. Attempting to write the object as-is.")
    writeH5AD(obj, out_file)
  }
}

In [ ]:
lueven_processed <- readRDS("objects/seurat_allsamples_allspots.rds")
lueven_processed[["visium"]] <- as(lueven_processed[["visium"]], Class = "Assay")
SaveH5Seurat(lueven_processed, filename = "lueven_TRUE", overwrite=T)
Convert("lueven_TRUE.h5seurat", dest = "h5ad", assay = "visium")

In [ ]:
# Reference remapping and prep

mart_df <- read.csv("objects/mart_export.txt", header = TRUE, sep = "\t", stringsAsFactors = FALSE)
colnames(mart_df)[1:3] <- c("ensembl_id", "gene_name", "mgi_symbol")
mart_df <- mart_df[!is.na(mart_df$mgi_symbol) & mart_df$mgi_symbol != "", ]

# Read scRNAseq reference H5AD
sce_ref_mart <- readH5AD("data/scRNA/reference_mapping/data/scRNA/adata_query.h5ad")
sc_count_mart <- assay(sce_ref_mart, "X")

# Remap rownames using mart_df
common_ids <- intersect(rownames(sc_count_mart), mart_df$ensembl_id)
mart_df_ordered <- mart_df[match(rownames(sc_count_mart), mart_df$ensembl_id), ]
new_row_names <- ifelse(
  is.na(mart_df_ordered$mgi_symbol),
  rownames(sc_count_mart),
  mart_df_ordered$mgi_symbol
)
rownames(sc_count_mart) <- new_row_names

# Extract meta
sc_meta_mart <- as.data.frame(colData(sce_ref_mart))[, c("cell_type", "donor"), drop = FALSE]
colnames(sc_meta_mart) <- c("cellType", "sampleInfo")
sc_meta_mart$cellID <- rownames(sc_meta_mart)
sc_meta_mart <- sc_meta_mart[, c("cellID", "cellType", "sampleInfo")]

In [ ]:
# Prep arguments for CARD - previously run multiple options, optimized for one used in analysis
scaleVec      <- c("lowres")
shapeSpotVec  <- c("Circle")
numCellVec    <- c(7)

seurat_obj <- readRDS("objects/seurat_allsamples_qcspots_processed.rds")
sce <- readH5AD("lueven_TRUE.h5ad")

counts_matrix <- assay(sce, "X")
all_cells <- colnames(counts_matrix)
donor_names <- unique(seurat_obj@meta.data$donor)

sc_count <- sc_count_mart
sc_meta  <- sc_meta_mart

ref_outdir <- "results/intermediate/CARD"

current_scale     <- "lowres"
current_shapeSpot <- "Circle"
current_numCell   <- 7

# Configuration-specific subfolder
cfg_outdir <- file.path(ref_outdir, paste0("output_scale_", current_scale,
                                           "_shapeSpot_", current_shapeSpot,
                                           "_numCell_", current_numCell
                                          )
                        )

if(!dir.exists(cfg_outdir)) {
  dir.create(cfg_outdir, recursive = TRUE)
}

message("Running configuration: ", basename(cfg_outdir))

In [ ]:
# Run CARD - only step 1 of initial run required, no visualization to safe time

for(donor_name in donor_names) {
  message("Processing donor: ", donor_name)
  donor_layer <- subset(seurat_obj, subset = donor == donor_name)
  
  coords_df <- GetTissueCoordinates(donor_layer, scale = current_scale, cols = c("imagerow", "imagecol"))
  rownames(coords_df) <- coords_df$cell
  coords_df$cell <- NULL
  
  counts_layer <- counts_matrix[, rownames(coords_df)]
    
  # Create the CARD object with the chosen reference sc_count and sc_meta
  CARD_obj <- createCARDObject(
    sc_count         = sc_count,
    sc_meta          = sc_meta,
    spatial_count    = counts_layer,
    spatial_location = coords_df,
    ct.varname       = "cellType",
    ct.select        = unique(sc_meta$cellType),
    sample.varname   = "sampleInfo",
    minCountGene     = 100,
    minCountSpot     = 5
  )

  # Run deconvolution (time-intensive)  
  CARD_obj <- CARD_deconvolution(CARD_object = CARD_obj)
  all_ct      <- unique(sc_meta$cellType)
  res_CARD    <- CARD_obj@Proportion_CARD
  ct.present  <- intersect(all_ct, colnames(res_CARD))
  if (length(ct.present)==0) {
      stop("None of your metadata cellType values are in the deconvolution result!")
  }

  saveRDS(CARD_obj, file = file.path(cfg_outdir, paste0(donor_name, "_CARDobj_step1.Rds"))) 
}  

In [ ]:
# Convert CARD to anndata to process in Python
base_dir <- "results/intermediate/CARD/output_scale_lowres_shapeSpot_Circle_numCell_7"
out_base_dir <- "results/intermediate/CARD_anndata"

all_files <- list.files(path = base_dir, pattern = "\\.Rds$", recursive = TRUE, full.names = TRUE)

file_patterns <- "(_CARDobj_step1.Rds)$"
files_to_convert <- all_files[grepl(file_patterns, all_files)]

# Loop through the selected files and convert each
for (file in files_to_convert) {
  rel_path <- substring(file, nchar(base_dir) + 2)
  new_file <- file.path(out_base_dir, sub("\\.Rds$", ".h5ad", rel_path))
  if (file.exists(new_file)) {
    message("Skipping conversion for ", file, " (", new_file, " already exists).")
    next
  }
  new_dir <- dirname(new_file)
  if (!dir.exists(new_dir)) {
    dir.create(new_dir, recursive = TRUE)
  }
 
  obj <- readRDS(file)
  convert_CARD_to_AnnData(obj, new_file)
}